In [0]:
CREATE OR REPLACE VIEW smart_meters.gold.vw_night_owls AS
WITH daily_metrics AS (
  SELECT 
    dim_household_lclid,
    t.year as year, 
    t.month as month, 
    t.day as day,
    SUM(energy_spent) AS daily_total,
    SUM(CASE WHEN hour >= 22 OR hour < 7 THEN energy_spent ELSE 0 END) AS night_total,
    COUNT(DISTINCT hour) AS hours_recorded
  FROM smart_meters.gold.fact_consumption
  INNER JOIN smart_meters.gold.dim_time as t 
    ON dim_time_id = t.id
  GROUP BY dim_household_lclid, year, month, day
)
SELECT 
  dim_household_lclid,
  COUNT(*) AS total_days,
  COUNT(CASE WHEN hours_recorded = 24 AND (night_total / NULLIF(daily_total, 0)) > 0.4 
             THEN 1 END) AS night_owl_days,
  ROUND(AVG(CASE WHEN hours_recorded = 24 
                 THEN (night_total / NULLIF(daily_total, 0)) END) * 100, 2) AS avg_night_percentage,
  CASE 
    WHEN COUNT(CASE WHEN hours_recorded = 24 AND (night_total / NULLIF(daily_total, 0)) > 0.4 THEN 1 END) > 0
         AND ROUND(AVG(CASE WHEN hours_recorded = 24 THEN (night_total / NULLIF(daily_total, 0)) END) * 100, 2) > 40
    THEN TRUE 
    ELSE FALSE 
  END AS is_night_owl
FROM daily_metrics
GROUP BY dim_household_lclid
ORDER BY night_owl_days DESC;

-- SELECT * FROM smart_meters.gold.vw_night_owls WHERE is_night_owl = TRUE;
SELECT COUNT(*) FROM smart_meters.gold.vw_night_owls WHERE is_night_owl = TRUE;